In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot"

os.environ["HADOOP_HOME"] = r"C:\hadoop"

os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print(
    "winutils exists =",
    os.path.exists(r"C:\hadoop\bin\winutils.exe"),
)
print(
    "hadoop.dll exists =",
    os.path.exists(r"C:\hadoop\bin\hadoop.dll"),
)

JAVA_HOME = C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot
HADOOP_HOME = C:\hadoop
winutils exists = True
hadoop.dll exists = True


In [2]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [3]:
!set PYTHON_GIL=0

In [4]:
%load_ext autoreload
%autoreload 2

import os

# 1. Force single-threading ONLY for the import phase
os.environ["OMP_NUM_THREADS"] = "1"
from credit_risk import run_pipeline

# 2. Immediately restore multi-threading for your training process
# Set this to your actual CPU physical core count (e.g., "4", "8", "12")
os.environ["OMP_NUM_THREADS"] = "16"

### Setup Path

In [5]:
from pathlib import Path
import os

if "project_path" not in globals():
    project_path = Path.cwd().parent
    os.chdir(project_path)

print("Project path:", project_path)

Project path: c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


### Run Main Pipeline

In [6]:
import os
import time
import threading
import psutil


def monitor_resources(interval=5):

    current_pid = os.getpid()

    while True:
        try:
            memory = psutil.virtual_memory()

            python_ram = psutil.Process(current_pid).memory_info().rss / (1024**3)

            java_processes = []

            for proc in psutil.process_iter(["pid", "name", "memory_info"]):
                try:
                    name = (proc.info["name"] or "").lower()

                    if "java" in name:
                        ram = proc.info["memory_info"].rss / (1024**3)

                        java_processes.append((proc.info["pid"], ram))

                except (
                    psutil.NoSuchProcess,
                    psutil.AccessDenied,
                ):
                    continue

            java_ram = sum(ram for _, ram in java_processes)

            print(
                f"[MONITOR] "
                f"System={memory.percent:.1f}% "
                f"Available={memory.available / (1024**3):.1f}GB | "
                f"Python={python_ram:.1f}GB | "
                f"Java={java_ram:.1f}GB | "
                f"CPU={psutil.cpu_percent(interval=1):.1f}%"
            )

            time.sleep(interval)

        except Exception as e:
            print(f"[MONITOR] {e}")

            break

In [7]:
run_pipeline(project_path)

21:50:17  INFO      utils.spark   Creating Spark session: master=local[16] driver_memory=12g shuffle_partitions=256
c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
21:50:23  INFO      utils.spark   Spark session created: version=4.2.0 default_parallelism=256
21:50:23  INFO      pipeline      ━━ Pipeline started ━━ project=C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk
21:50:23  INFO      ingest        Ingestion skipped by configuration
21:50:23  INFO      data_preprocess  Preprocessing skipped by configuration
21:50:23  INFO      reporting     Data-quality reporting skipped by configuration
21:50:23  INFO      modelling     Modelling pipeline skipped by configuration
21:50:23  INFO      modelling.artifacts_spark  YAML a